In [0]:
from pyspark.sql.functions import col, count, sum, when, to_date, explode, sequence, coalesce, current_date, lit, sum, when, countDistinct
from pyspark.sql import Row
from functools import reduce

In [0]:
catalog = "subscription_platform"
silver_schema = "silver"

silver_events = f"{catalog}.{silver_schema}.subscription_events_clean"
silver_subscriptions_history = f"{catalog}.{silver_schema}.subscriptions_history"

In [0]:
history_df = spark.read.table(silver_subscriptions_history)
events_df = spark.read.table(silver_events)

In [0]:
gold_schema = "gold"


### Gold -> Daily Activity Metrics

In [0]:
gold_daily_activity_metrics_table = f"{catalog}.{gold_schema}.daily_activity_metrics"
gold_daily_activity_metrics_path = "s3://subscription-revenue-platform/gold/daily_revenue_metrics/"

In [0]:
new_subs = (
    events_df
    .filter(col("event_type") == "subscription_created")
    .groupBy(col("event_date").alias("metric_date"))
    .agg(count("*").alias("new_subscriptions"))
)

display(new_subs)

metric_date,new_subscriptions
2026-04-06,2
2026-04-03,6
2026-04-10,3
2026-04-09,3
2026-04-07,9
2026-04-15,8
2026-04-14,12
2026-04-02,5
2026-04-04,2
2026-04-05,4


In [0]:
cancellations = (
    events_df
    .filter(col("event_type") == "subscription_cancelled")
    .groupBy(col("event_date").alias("metric_date"))
    .agg(count("*").alias("cancellations"))
)

display(cancellations)

metric_date,cancellations
2026-04-03,4
2026-04-12,11
2026-04-16,4
2026-04-13,4
2026-04-11,8
2026-04-15,3
2026-04-08,7
2026-04-01,4
2026-04-06,6
2026-04-10,8


In [0]:
plan_changes = (
    events_df
    .filter(col("event_type") == "plan_changed")
    .groupBy(col("event_date").alias("metric_date"))
    .agg(count("*").alias("plan_changes"))
)

display(plan_changes)

metric_date,plan_changes
2026-04-07,4
2026-04-08,6
2026-04-12,4
2026-04-02,7
2026-04-11,7
2026-04-15,4
2026-04-13,6
2026-04-01,3
2026-04-14,8
2026-04-16,1


In [0]:
dfs = [new_subs, cancellations, plan_changes]

daily_activity_metrics = reduce(
    lambda left, right: left.join(right, on="metric_date", how="outer"),
    dfs
).fillna(0)

display(daily_activity_metrics)

metric_date,new_subscriptions,cancellations,plan_changes
2026-04-16,1,4,1
2026-03-31,0,1,2
2026-04-01,1,4,3
2026-04-06,2,6,3
2026-04-11,6,8,7
2026-04-08,9,7,6
2026-04-10,3,8,5
2026-04-09,3,2,5
2026-04-07,9,4,4
2026-04-14,12,8,8


In [0]:
(
daily_activity_metrics.write
    .format("delta")
    .mode("append")
    .saveAsTable(gold_daily_activity_metrics_table)
)

In [0]:
%sql

SELECT *
FROM subscription_platform.gold.daily_activity_metrics
ORDER BY metric_date;

metric_date,new_subscriptions,cancellations,plan_changes
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2
2026-03-31,0,1,2



### Gold -> Subscription KPIs Snapshot

In [0]:
gold_subscription_kpis_snapshot_table = f"{catalog}.{gold_schema}.subscription_kpis_snapshot"
gold_subscription_kpis_snapshot_path = "s3://subscription-revenue-platform/gold/subscription_kpis_snapshot/"

In [0]:
dbutils.widgets.text("snapshot_date", "")
snapshot_date = dbutils.widgets.get("snapshot_date")

In [0]:
if snapshot_date == "":
    raise ValueError("snapshot_date parameter is required")

In [0]:
snapshot_df = (
    history_df
    .agg(
        sum(when((col("is_current") == True) & (col("status") == "active"), col("amount"))).alias("mrr"),
        sum(when((col("is_current") == True) & (col("status") == "active"), 1).otherwise(0)).alias("active_subscriptions"),
        sum(when((col("is_current") == True) & (col("status") == "cancelled"), 1).otherwise(0)).alias("cancelled_subscriptions"),
        countDistinct(col("subscription_id")).alias("total_subscriptions")
    )
    .withColumn("snapshot_date", to_date(lit(snapshot_date)))
    .select(
        "snapshot_date",
        "mrr",
        col("active_subscriptions").cast("int"),
        col("cancelled_subscriptions").cast("int"),
        col("total_subscriptions").cast("int")
    )
)

display(snapshot_df)

snapshot_date,mrr,active_subscriptions,cancelled_subscriptions,total_subscriptions
2026-04-25,8000.0,5,4,9


In [0]:
snapshot_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(gold_subscription_kpis_snapshot_table)

In [0]:
%sql

SELECT *
FROM subscription_platform.gold.subscription_kpis_snapshot
ORDER BY snapshot_date;

snapshot_date,mrr,active_subscriptions,cancelled_subscriptions,total_subscriptions
2025-04-10,3000.0,2,4,6
2025-04-11,9000.0,8,1,9
2025-04-12,10000.0,8,1,9
2026-04-13,9000.0,8,1,9
2026-04-14,11000.0,7,2,9
2026-04-15,11000.0,8,1,9
2026-04-16,9000.0,8,1,9
2026-04-17,12000.0,8,1,9
2026-04-18,9000.0,8,1,9
2026-04-19,11000.0,7,2,9
